# Final experiments: informal road detection in Yaou and Bonoua
Runs every result for the final paper and saves each one to Google Drive as soon as it finishes.
If Colab disconnects, run all cells again: finished runs are skipped.

**Splits** (from the QGIS toolbar status and town):
- **Test** = status `test` (Bonoua). Scored once, at the end of each run.
- **Validation** = status `val` (a separate Yaou strip). Used to pick the epoch and the threshold.
- **Corrected training** = status `reviewed`, never Bonoua.
- **OSM-only training** = status `auto`, never Bonoua.

**Before running**, put `Dataset.zip` in `MyDrive/road_data/`, containing `images/`, `masks/`, `masks_osm/`, `tiles_log.csv` and `labeling_log.csv`.
Runtime > Change runtime type > **L4 GPU** (or A100).

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount("/content/drive")
!pip -q install segmentation-models-pytorch tifffile scikit-image
import os, glob, csv, json, time, random, platform, numpy as np, pandas as pd, torch, tifffile
import segmentation_models_pytorch as smp, skimage, scipy
DATA_ZIP = "/content/drive/MyDrive/road_data/Dataset_v4.zip"
SAVE = "/content/drive/MyDrive/road_models"
RES = f"{SAVE}/final_v4"; os.makedirs(f"{RES}/figs", exist_ok=True); os.makedirs(f"{RES}/runs", exist_ok=True)
if not os.path.isdir("/content/data_v4"):
    !unzip -q -o "{DATA_ZIP}" -d /content/data_v4
root = "/content/data_v4" if os.path.isdir("/content/data_v4/images") else glob.glob("/content/data_v4/*/images")[0].rsplit("/images", 1)[0]
dev = "cuda"
env = {"python": platform.python_version(), "torch": torch.__version__, "segmentation_models_pytorch": smp.__version__,
       "numpy": np.__version__, "scipy": scipy.__version__, "scikit-image": skimage.__version__,
       "gpu": torch.cuda.get_device_name(0)}
json.dump(env, open(f"{RES}/environment.json", "w"), indent=1)
print(env, "\ndata:", root)

In [ ]:
# 2. Load tiles and build the splits
TEST_TOWNS = {"bonoua", "buffer"}  # never used for training
LOG = "/content/drive/MyDrive/road_data/tiles_log.csv"   # updated town tags, if uploaded
log = pd.read_csv(LOG if os.path.exists(LOG) else f"{root}/tiles_log.csv")
if "town" not in log: log["town"] = log["site"]
log["town"] = log["town"].fillna(log["site"]).str.lower()
log["file"] = log["image"].str.split("/").str[-1]
# Town tags come from QGIS "Set town". Any non-test tile within 1 km (4 tiles) of a test tile = buffer (excluded).
tx, ty = log.loc[log["status"] == "test", "gx"].values, log.loc[log["status"] == "test", "gy"].values
dist = np.array([np.sqrt((tx - x) ** 2 + (ty - y) ** 2).min() for x, y in zip(log["gx"], log["gy"])])
log["dist_to_test_km"] = dist * 0.256
log.loc[(log["status"] != "test") & (dist < 4), "town"] = "buffer"
log.loc[log["status"] == "test", "town"] = "bonoua"
print(log.groupby(["town", "status"]).agg(tiles=("gx", "size"), min_km_to_test=("dist_to_test_km", "min")))

def read(path):
    a = tifffile.imread(path)
    if a.ndim == 3 and a.shape[0] == 3: a = a.transpose(1, 2, 0)
    return a

tiles = {}
for _, r in log.iterrows():
    n = r["file"]
    if not all(os.path.exists(f"{root}/{d}/{n}") for d in ("images", "masks", "masks_osm")):
        continue
    fixed = (read(f"{root}/masks/{n}") > 0).astype(np.float32)
    osm = (read(f"{root}/masks_osm/{n}") > 0).astype(np.float32)
    st = str(r["status"])
    if st not in ("test", "val", "reviewed") and osm.mean() <= 0.005:   # uncorrected tile with no mapped road
        continue
    tiles[n] = dict(img=read(f"{root}/images/{n}")[..., :3].astype(np.uint8), fixed=fixed, osm=osm,
                    status=st, town=r["town"], gx=int(r["gx"]), gy=int(r["gy"]))

test_n = sorted(n for n, t in tiles.items() if t["status"] == "test")
val_n = sorted(n for n, t in tiles.items() if t["status"] == "val")
corr_n = sorted(n for n, t in tiles.items() if t["status"] == "reviewed" and t["town"] not in TEST_TOWNS)
other_n = sorted(n for n, t in tiles.items() if t["status"] == "auto" and t["town"] not in TEST_TOWNS)
train_all = corr_n + other_n
assert test_n, "No test tiles: mark them with the Test button in QGIS."
assert not any(tiles[n]["town"] in TEST_TOWNS for n in train_all), "A test town tile is in training."
if not val_n:   # fallback: hold out a strip of OSM-labelled training tiles (noisier, reported as such)
    cols = sorted({tiles[n]["gx"] for n in other_n}); strip = set(cols[:max(1, len(cols) // 8)])
    val_n = [n for n in other_n if tiles[n]["gx"] in strip]; other_n = [n for n in other_n if n not in val_n]
    train_all = corr_n + other_n
    print("WARNING: no 'val' tiles, using a held-out strip of OSM-labelled tiles for validation.")
ORDER_CORR = random.Random(0).sample(corr_n, len(corr_n))     # 5 corrected tiles are inside 10, etc.
ORDER_ALL = random.Random(0).sample(train_all, len(train_all))
split = pd.DataFrame([dict(tile=n, town=t["town"], status=t["status"],
                           split=("test" if n in test_n else "val" if n in val_n else
                                  "train_corrected" if n in corr_n else "train_osm" if n in other_n else "excluded"))
                      for n, t in tiles.items()])
split.to_csv(f"{RES}/splits.csv", index=False)
print(split.groupby(["split", "town"]).size())

In [ ]:
# ===== scoring, failure types, flags, statistics (numpy only) =====
import numpy as np
from scipy import ndimage
from skimage.morphology import skeletonize

PIX = 0.5            # metres per pixel
TOL = 4              # 2 m tolerance for relaxed scores and "found"
NEAR = 4             # <= 2 m from OSM centreline = correctly placed
FAR = 30             # > 15 m from OSM = missing / phantom
FLAG_GAP = 6         # flags: predicted road more than 3 m from any OSM road pixel
FLAG_MIN = 100       # flags: at least 25 m2
SEG_MIN = 20         # missing-road segment: at least 10 m of centreline

def counts(p, t):
    """Per-tile pixel counts for strict and relaxed scores."""
    p, t = p > 0, t > 0
    tdil = ndimage.binary_dilation(t, iterations=TOL) if t.any() else t
    pdil = ndimage.binary_dilation(p, iterations=TOL) if p.any() else p
    return dict(tp=int((p & t).sum()), fp=int((p & ~t).sum()), fn=int((~p & t).sum()),
                rp=int((p & tdil).sum()), np=int(p.sum()), rt=int((t & pdil).sum()), nt=int(t.sum()))

def score(c):
    """Scores from summed counts (dict or DataFrame row sums)."""
    tp, fp, fn = c["tp"], c["fp"], c["fn"]
    P, R = tp / (tp + fp + 1e-9), tp / (tp + fn + 1e-9)
    rP, rR = c["rp"] / (c["np"] + 1e-9), c["rt"] / (c["nt"] + 1e-9)
    return dict(precision=P, recall=R, f1=2*P*R/(P+R+1e-9), iou=tp/(tp+fp+fn+1e-9),
                relaxed_precision=rP, relaxed_recall=rR, relaxed_f1=2*rP*rR/(rP+rR+1e-9))

def _skel(m):
    m = m > 0
    return skeletonize(m) if m.any() else m

def _dist(m):
    """Distance in pixels to the nearest True pixel of m (large if m is empty)."""
    m = m > 0
    return ndimage.distance_transform_edt(~m) if m.any() else np.full(m.shape, 1e6)

def failure_types(osm, fixed, pred=None):
    """Compare OSM with corrected labels along road centrelines (lengths in metres).
    matched   : real road with an OSM line within 2 m
    misplaced : real road with the nearest OSM line 2-15 m away (OSM offset)
    missing   : real road with no OSM line within 15 m
    phantom   : OSM line with no real road within 15 m
    If pred is given, also returns how much of each type the model recovers (within 2 m)."""
    cs, os_ = _skel(fixed), _skel(osm)
    d_osm, d_fix = _dist(os_), _dist(cs)
    dv = d_osm[cs]
    out = dict(real_m=cs.sum() * PIX, osm_m=os_.sum() * PIX,
               matched_m=(dv <= NEAR).sum() * PIX,
               misplaced_m=((dv > NEAR) & (dv <= FAR)).sum() * PIX,
               missing_m=(dv > FAR).sum() * PIX,
               phantom_m=(d_fix[os_] > FAR).sum() * PIX,
               offsets_m=list((dv[(dv > NEAR) & (dv <= FAR)] * PIX).round(2)))
    if pred is not None:
        hit = _dist(pred)[cs] <= TOL
        for name, sel in [("matched", dv <= NEAR), ("misplaced", (dv > NEAR) & (dv <= FAR)),
                          ("missing", dv > FAR)]:
            out[f"model_found_{name}_m"] = (hit & sel).sum() * PIX
    return out

def flags_for_tile(pred, osm, fixed):
    """Flags = predicted road > 3 m from any OSM road pixel, pieces >= 25 m2.
    Each flag is classified against the corrected labels:
      missing_road   : on a real road with no OSM line within 15 m
      misplaced_osm  : on a real road that OSM draws 2-15 m away
      not_a_road     : no real road under the flag
    Also returns the real missing-road segments and whether a flag touched them (flag recall)."""
    p, o, f = pred > 0, osm > 0, fixed > 0
    cand = p & (_dist(o) > FLAG_GAP)
    lab, k = ndimage.label(cand, structure=np.ones((3, 3)))
    f_near = ndimage.binary_dilation(f, iterations=2) if f.any() else f
    d_osm_c = _dist(_skel(o))
    flags = []
    for i in range(1, k + 1):
        comp = lab == i
        if comp.sum() < FLAG_MIN:
            continue
        on_road = (comp & f_near).sum() / comp.sum() >= 0.3
        if not on_road:
            kind = "not_a_road"
        else:
            kind = "missing_road" if np.median(d_osm_c[comp & f_near]) > FAR else "misplaced_osm"
        ys, xs = np.nonzero(comp)
        flags.append(dict(kind=kind, area_m2=comp.sum() * PIX * PIX, row=int(ys.mean()), col=int(xs.mean())))
    cs = _skel(f)
    miss = cs & (_dist(_skel(o)) > FAR)
    mlab, mk = ndimage.label(miss, structure=np.ones((3, 3)))
    flagmask = np.isin(lab, [i for i in range(1, k + 1) if (lab == i).sum() >= FLAG_MIN])
    d_flag = _dist(flagmask)
    segs = []
    for i in range(1, mk + 1):
        seg = mlab == i
        if seg.sum() < SEG_MIN:
            continue
        segs.append(dict(length_m=seg.sum() * PIX, flagged=bool((d_flag[seg] <= TOL).any())))
    return flags, segs

def wilson(k, n, z=1.96):
    """95% Wilson confidence interval for a proportion k/n."""
    if n == 0:
        return (float("nan"), float("nan"), float("nan"))
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (p, max(0.0, c - h), min(1.0, c + h))

def bootstrap_f1_diff(a, b, n_boot=2000, seed=0):
    """a, b: per-tile count arrays (n_tiles x 3: tp, fp, fn) for two models on the same tiles.
    Returns F1(a) - F1(b) and its 95% bootstrap CI over tiles."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    def f1(x):
        tp, fp, fn = x.sum(0)
        return 2*tp / (2*tp + fp + fn + 1e-9)
    rng = np.random.default_rng(seed); n = len(a); diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs.append(f1(a[idx]) - f1(b[idx]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return f1(a) - f1(b), lo, hi

def cohens_d(x, y):
    """Effect size between two sets of seed scores."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    s = np.sqrt(((len(x)-1)*x.var(ddof=1) + (len(y)-1)*y.var(ddof=1)) / max(1, len(x)+len(y)-2))
    return float((x.mean() - y.mean()) / s) if s > 0 else float("nan")

In [ ]:
# 4. Training with model selection on the validation tiles
MEAN = np.array([0.485, 0.456, 0.406]); STD = np.array([0.229, 0.224, 0.225])   # ImageNet stats, RGB order
def to_x(img): return ((img / 255.0 - MEAN) / STD).transpose(2, 0, 1)
EPOCHS = 20

class DS(torch.utils.data.Dataset):
    def __init__(self, items): self.items = items                      # (img, mask, clean)
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        img, m, clean = self.items[i]
        k = np.random.randint(4); img, m = np.rot90(img, k), np.rot90(m, k)
        if np.random.rand() < .5: img, m = img[:, ::-1], m[:, ::-1]
        return (torch.tensor(to_x(img).copy(), dtype=torch.float32),
                torch.tensor(m.copy())[None], torch.tensor(float(clean)))

def masked_loss(out, y, w):
    bce = torch.nn.functional.binary_cross_entropy_with_logits(out, y, reduction="none")
    bce = (bce * w).sum() / w.sum().clamp(min=1)
    p = torch.sigmoid(out) * w; yy = y * w
    return bce + 1 - (2 * (p * yy).sum() + 1) / (p.sum() + yy.sum() + 1)

def probs(model, names):
    model.eval(); out = []
    with torch.no_grad():
        for n in names:
            x = torch.tensor(to_x(tiles[n]["img"]).copy(), dtype=torch.float32)[None].to(dev)
            out.append(torch.sigmoid(model(x))[0, 0].float().cpu().numpy())
    return out

def val_f1(model, thr=0.5):
    c = {k: 0 for k in ("tp", "fp", "fn")}
    for p, n in zip(probs(model, val_n), val_n):
        pp, tt = p > thr, tiles[n]["fixed"] > 0
        c["tp"] += (pp & tt).sum(); c["fp"] += (pp & ~tt).sum(); c["fn"] += (~pp & tt).sum()
    return 2 * c["tp"] / (2 * c["tp"] + c["fp"] + c["fn"] + 1e-9)

def train(model, items, epochs, lr, method, seed, history):
    g = torch.Generator(); g.manual_seed(seed)
    dl = torch.utils.data.DataLoader(DS(items), batch_size=4, shuffle=True, drop_last=len(items) > 4, generator=g)
    opt = torch.optim.AdamW(model.parameters(), lr)
    best, best_state = -1, None
    for ep in range(epochs):
        model.train(); tot = 0
        for x, y, clean in dl:
            x, y, clean = x.to(dev), y.to(dev), clean.to(dev)
            out = model(x); w = torch.ones_like(y)
            if method == "ignore" and ep >= 5:          # no penalty for confident roads OSM is missing
                with torch.no_grad():
                    w[(y == 0) & (torch.sigmoid(out) > 0.8) & (clean == 0).view(-1, 1, 1, 1)] = 0
            loss = masked_loss(out, y, w)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        f = val_f1(model); history.append(dict(epoch=len(history) + 1, loss=tot / max(1, len(dl)), val_f1=float(f)))
        if f > best: best, best_state = f, {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)                   # best epoch chosen on validation only
    return model

from scipy import ndimage as _ndi
LAST_BLOB = 0
def clean(mask, min_px):
    """Drop connected road blobs smaller than min_px pixels (0.25 m2 each)."""
    if min_px <= 0: return mask
    lab, n = _ndi.label(mask)
    if n == 0: return mask
    sizes = np.bincount(lab.ravel()); keep = sizes >= min_px; keep[0] = False
    return keep[lab]

def pick_threshold(model):
    """Threshold and minimum blob size chosen together on validation only."""
    global LAST_BLOB
    pv = probs(model, val_n); tv = [tiles[n]["fixed"] > 0 for n in val_n]
    best = (0.5, 0, -1)
    for thr in np.arange(0.2, 0.75, 0.05):
        for mb in (0, 100, 200, 400):          # 0, 25, 50, 100 m2
            tp = fp = fn = 0
            for p, t in zip(pv, tv):
                pp = clean(p > thr, mb)
                tp += (pp & t).sum(); fp += (pp & ~t).sum(); fn += (~pp & t).sum()
            f = 2 * tp / (2 * tp + fp + fn + 1e-9)
            if f > best[2]: best = (float(round(thr, 2)), mb, f)
    LAST_BLOB = best[1]
    return best[0]

def new_model(backbone="resnet34", init="imagenet"):
    m = smp.DeepLabV3Plus(backbone, encoder_weights="imagenet", classes=1).to(dev)
    if init == "deepglobe":
        m.load_state_dict(torch.load(f"{SAVE}/dg_full_best.pt", map_location=dev))
    return m

In [ ]:
# 5. Evaluate any model on the test tiles (touched once per run)
def evaluate_test(model, thr):
    preds = [clean(p > thr, LAST_BLOB) for p in probs(model, test_n)]
    rows, ft_rows = [], []
    for n, p in zip(test_n, preds):
        t = tiles[n]
        rows.append(dict(tile=n, town=t["town"], **counts(p, t["fixed"])))
        ft = failure_types(t["osm"], t["fixed"], p); ft.pop("offsets_m")
        ft_rows.append(dict(tile=n, town=t["town"], **ft))
    return pd.DataFrame(rows), pd.DataFrame(ft_rows), preds

def summarise(df):
    s = score(df[["tp", "fp", "fn", "rp", "np", "rt", "nt"]].sum())
    per_town = {town: score(d[["tp", "fp", "fn", "rp", "np", "rt", "nt"]].sum()) for town, d in df.groupby("town")}
    return s, per_town

# Baseline A: OpenStreetMap itself, on every corrected tile, per town (RQ1)
osm_rows, ft_all = [], []
for n, t in tiles.items():
    if t["status"] in ("test", "val", "reviewed"):
        osm_rows.append(dict(tile=n, town=t["town"], status=t["status"], **counts(t["osm"], t["fixed"])))
        ft = failure_types(t["osm"], t["fixed"]); offs = ft.pop("offsets_m")
        ft_all.append(dict(tile=n, town=t["town"], status=t["status"], median_offset_m=np.median(offs) if offs else np.nan, **ft))
osm_df, ft_df = pd.DataFrame(osm_rows), pd.DataFrame(ft_all)
osm_df.to_csv(f"{RES}/osm_per_tile.csv", index=False); ft_df.to_csv(f"{RES}/failure_types_per_tile.csv", index=False)
osm_town = pd.DataFrame({town: score(d[["tp", "fp", "fn", "rp", "np", "rt", "nt"]].sum()) for town, d in osm_df.groupby("town")}).T
osm_town["tiles"] = osm_df.groupby("town").size(); osm_town.round(3).to_csv(f"{RES}/osm_score_per_town.csv")
print("OpenStreetMap vs corrected labels, per town\n", osm_town.round(3))

# Failure types per town (RQ2): share of real road length, typical OSM offset
g = ft_df.groupby("town")[["real_m", "osm_m", "matched_m", "misplaced_m", "missing_m", "phantom_m"]].sum()
ft_town = pd.DataFrame({
    "real_road_km": g.real_m / 1000, "osm_road_km": g.osm_m / 1000,
    "matched_%": 100 * g.matched_m / g.real_m, "misplaced_%": 100 * g.misplaced_m / g.real_m,
    "missing_%": 100 * g.missing_m / g.real_m, "phantom_%_of_osm": 100 * g.phantom_m / g.osm_m,
    "median_offset_m_misplaced": ft_df.groupby("town").median_offset_m.median()})
ft_town.round(2).to_csv(f"{RES}/failure_types_per_town.csv"); print("\nFailure types per town\n", ft_town.round(2))

# Labelling cost
if os.path.exists(f"{root}/labeling_log.csv"):
    lab = pd.read_csv(f"{root}/labeling_log.csv")
    km2 = len(lab) * (256 * 256) / 1e6
    cost = dict(tiles_timed=len(lab), total_minutes=float(lab.minutes.sum()),
                minutes_per_tile=float(lab.minutes.mean()), hours_per_km2=float(lab.minutes.sum() / 60 / km2) if km2 else None)
    json.dump(cost, open(f"{RES}/labeling_cost.json", "w"), indent=1); print("\nLabelling cost:", cost)

In [ ]:
# 6. Baseline B: full DeepGlobe model, no local training (threshold chosen on validation)
if not os.path.exists(f"{RES}/runs/deepglobe_only.json") and os.path.exists(f"{SAVE}/dg_full_best.pt"):
    m = new_model(init="deepglobe"); thr = pick_threshold(m)
    df, ftd, _ = evaluate_test(m, thr); s, pt = summarise(df)
    df.to_csv(f"{RES}/runs/deepglobe_only_tiles.csv", index=False); ftd.to_csv(f"{RES}/runs/deepglobe_only_ft.csv", index=False)
    json.dump(dict(name="deepglobe_only", threshold=thr, min_blob_px=LAST_BLOB, **s, per_town=pt), open(f"{RES}/runs/deepglobe_only.json", "w"), indent=1)
    print("DeepGlobe only:", {k: round(v, 3) for k, v in s.items()})

In [ ]:
# 7. All training runs (most important first; results saved after each run)
def run(k, method="normal", backbone="resnet34", init="imagenet", frac=1.0, seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    clean = set(ORDER_CORR[:k])
    pool = sorted(set(ORDER_ALL[:max(1, round(len(ORDER_ALL) * frac))]) | clean) if frac < 1 else train_all
    model, hist = new_model(backbone, init), []
    if method == "finetune" and k > 0:       # OSM labels first, then corrected tiles only
        train(model, [(tiles[n]["img"], tiles[n]["osm"], 0) for n in pool], EPOCHS, 1e-4, "normal", seed, hist)
        train(model, [(tiles[n]["img"], tiles[n]["fixed"], 1) for n in clean], 8, 5e-5, "normal", seed, hist)
    else:
        items = [(tiles[n]["img"], tiles[n]["fixed"] if n in clean else tiles[n]["osm"], int(n in clean)) for n in pool]
        train(model, items, EPOCHS, 1e-4 if init == "imagenet" else 5e-5, method, seed, hist)
    thr = pick_threshold(model)
    df, ftd, preds = evaluate_test(model, thr)
    return model, thr, df, ftd, hist

n_corr = len(corr_n)
FAST = True        # True: about 1-1.5 h (core results). False: the full plan (about 3 h)
if FAST:
    EPOCHS = 40        # validation picks the best epoch
    K_LIST = sorted({k for k in (0, 5, 10) if k <= n_corr} | {n_corr})
    K_MAX, SEEDS = n_corr, [0, 1, 2]
    plan = [dict(group="corrected_tiles", k=k, seed=s) for k in K_LIST for s in SEEDS]
    if os.path.exists(f"{SAVE}/dg_full_best.pt"):
        plan += [dict(group="pretraining", k=K_MAX, init="deepglobe", seed=s) for s in SEEDS]
    plan += [dict(group="methods", k=K_MAX, method=m, seed=s) for m in ("finetune", "ignore") for s in SEEDS]
    plan += [dict(group="backbone", k=K_MAX, backbone=b, seed=0) for b in ("resnet50", "efficientnet-b4")]
    plan += [dict(group="training_size", k=K_MAX, frac=f, seed=s) for f in (0.25, 0.5, 0.75) for s in SEEDS[:2]]
else:
    K_LIST = sorted({k for k in (0, 5, 10, 20, 40) if k <= n_corr} | {n_corr})
    K_MAX, SEEDS = n_corr, [0, 1, 2]
    plan = [dict(group="corrected_tiles", k=k, seed=s) for k in K_LIST for s in SEEDS]
    if os.path.exists(f"{SAVE}/dg_full_best.pt"):
        plan += [dict(group="pretraining", k=K_MAX, init="deepglobe", seed=s) for s in SEEDS]
    plan += [dict(group="methods", k=k, method=m, seed=s) for m in ("finetune", "ignore") for k in K_LIST
             if not (m == "finetune" and k == 0) for s in SEEDS]
    plan += [dict(group="training_size", k=K_MAX, frac=f, seed=s) for f in (0.25, 0.5, 0.75) for s in SEEDS]
    plan += [dict(group="backbone", k=K_MAX, backbone=b, seed=s) for b in ("resnet50", "efficientnet-b4") for s in SEEDS]

def rid(c): return "_".join(f"{k}-{c[k]}" for k in sorted(c) if k != "group").replace(".", "p")
todo = [c for c in plan if not os.path.exists(f"{RES}/runs/{rid(c)}.json")]
print(f"{len(plan)} runs planned, {len(todo)} to do")
for i, c in enumerate(todo):
    t0 = time.time(); cfg = {k: v for k, v in c.items() if k != "group"}
    model, thr, df, ftd, hist = run(**cfg)
    s, pt = summarise(df)
    df.to_csv(f"{RES}/runs/{rid(c)}_tiles.csv", index=False); ftd.to_csv(f"{RES}/runs/{rid(c)}_ft.csv", index=False)
    pd.DataFrame(hist).to_csv(f"{RES}/runs/{rid(c)}_trainlog.csv", index=False)
    full_cfg = {**dict(method="normal", backbone="resnet34", init="imagenet", frac=1.0), **cfg}
    full_cfg.update(epochs=EPOCHS, batch_size=4, optimizer="AdamW", lr=1e-4, loss="BCE + Dice",
                    augmentation="rot90 + horizontal flip", normalization="ImageNet mean/std, RGB")
    json.dump(dict(name=rid(c), group=c["group"], config=full_cfg, threshold=thr, min_blob_px=LAST_BLOB,
                   minutes=round((time.time() - t0) / 60, 1), **s, per_town=pt),
              open(f"{RES}/runs/{rid(c)}.json", "w"), indent=1)
    if c == dict(group="corrected_tiles", k=K_MAX, seed=0):
        torch.save(model.state_dict(), f"{RES}/main_model.pt")
    print(f"[{i+1}/{len(todo)}] {rid(c)}  F1 {s['f1']:.3f}  recall {s['recall']:.3f}  precision {s['precision']:.3f}  thr {thr}  ({(time.time()-t0)/60:.1f} min)")

In [ ]:
# 8. Results tables, statistics and charts
import matplotlib.pyplot as plt
runs = [json.load(open(p)) for p in glob.glob(f"{RES}/runs/*.json")]
R = pd.DataFrame([{**r["config"], "name": r["name"], "group": r["group"], "threshold": r["threshold"],
                   **{m: r[m] for m in ("precision", "recall", "f1", "iou", "relaxed_precision", "relaxed_recall", "relaxed_f1")}}
                  for r in runs if "config" in r])
R.to_csv(f"{RES}/all_runs.csv", index=False)
cols = ["precision", "recall", "f1", "iou", "relaxed_f1"]
base = R[(R.method == "normal") & (R.backbone == "resnet34") & (R.init == "imagenet") & (R.frac == 1.0)]
def ms(d, by): return d.groupby(by)[cols].agg(["mean", "std"]).round(3)
tables = {
  "corrected_tiles_by_method": ms(pd.concat([base, R[R.group == "methods"]]), ["method", "k"]),
  "pretraining": ms(pd.concat([base[base.k == K_MAX], R[R.group == "pretraining"]]), "init"),
  "training_size": ms(pd.concat([base[base.k == K_MAX], R[R.group == "training_size"]]), "frac"),
  "backbone": ms(pd.concat([base[base.k == K_MAX], R[R.group == "backbone"]]), "backbone"),
}
with pd.ExcelWriter(f"{RES}/summary.xlsx") as xw:
    for name, t in tables.items():
        if len(t): print(f"\n{name}\n", t); t.to_excel(xw, name[:31])
    osm_town.round(3).to_excel(xw, "osm_per_town"); ft_town.round(2).to_excel(xw, "failure_types")

# Bootstrap CI over test tiles + effect size: main model vs each baseline
def tile_counts(names):
    d = pd.concat([pd.read_csv(f"{RES}/runs/{n}_tiles.csv") for n in names])
    return d.groupby("tile")[["tp", "fp", "fn"]].mean().sort_index()
main = base[base.k == K_MAX].name.tolist()
A = tile_counts(main)
refs = {"OpenStreetMap": osm_df[osm_df.status == "test"].set_index("tile")[["tp", "fp", "fn"]].sort_index(),
        "OSM labels only (0 corrected)": tile_counts(base[base.k == 0].name.tolist())}
if os.path.exists(f"{RES}/runs/deepglobe_only_tiles.csv"):
    refs["DeepGlobe only"] = pd.read_csv(f"{RES}/runs/deepglobe_only_tiles.csv").set_index("tile")[["tp", "fp", "fn"]].sort_index()
stats = []
for name, B in refs.items():
    common = A.index.intersection(B.index)
    d, lo, hi = bootstrap_f1_diff(A.loc[common].values, B.loc[common].values)
    stats.append(dict(baseline=name, f1_gain=round(d, 3), ci95_low=round(lo, 3), ci95_high=round(hi, 3)))
for grp, col in [("pretraining", "init"), ("backbone", "backbone")]:
    for v, d in R[R.group == grp].groupby(col):
        stats.append(dict(baseline=f"{grp}: {v} vs default", cohens_d=round(cohens_d(d.f1, base[base.k == K_MAX].f1), 2),
                          f1_gain=round(d.f1.mean() - base[base.k == K_MAX].f1.mean(), 3)))
stats = pd.DataFrame(stats); stats.to_csv(f"{RES}/significance.csv", index=False); print("\n", stats)

# Figure 1: F1 vs corrected tiles (and vs labelling hours), with OSM and DeepGlobe reference lines
fig, ax = plt.subplots(figsize=(7, 4.5))
for m, d in pd.concat([base, R[R.group == "methods"]]).groupby("method"):
    s = d.groupby("k").f1.agg(["mean", "std"])
    ax.errorbar(s.index, s["mean"], yerr=s["std"], marker="o", capsize=4, label={"normal": "Normal training", "finetune": "OSM first, then corrected", "ignore": "Ignore confident OSM gaps"}[m])
ax.axhline(osm_town.loc[list({tiles[n]['town'] for n in test_n})].f1.mean(), ls="--", c="gray", label="OpenStreetMap itself")
if os.path.exists(f"{RES}/runs/deepglobe_only.json"):
    ax.axhline(json.load(open(f"{RES}/runs/deepglobe_only.json"))["f1"], ls=":", c="k", label="DeepGlobe only")
ax.set_xlabel("Hand-corrected training tiles"); ax.set_ylabel("Test F1 (Bonoua, corrected labels)")
if os.path.exists(f"{RES}/labeling_cost.json"):
    mpt = json.load(open(f"{RES}/labeling_cost.json"))["minutes_per_tile"]
    sec = ax.secondary_xaxis("top", functions=(lambda k: k * mpt / 60, lambda h: h * 60 / mpt)); sec.set_xlabel("Labelling hours")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(f"{RES}/figs/corrected_tiles.png", dpi=200)

# Figure 2: ablations
fig, axs = plt.subplots(1, 3, figsize=(13, 4))
for a, (name, col) in zip(axs, [("pretraining", "init"), ("training_size", "frac"), ("backbone", "backbone")]):
    d = pd.concat([base[base.k == K_MAX], R[R.group == name]]).groupby(col).f1.agg(["mean", "std"]).fillna(0)
    a.bar(d.index.astype(str), d["mean"], yerr=d["std"], capsize=5, color="#1E6B57")
    a.set_title(name.replace("_", " ").title()); a.set_ylabel("Test F1"); a.set_ylim(0, max(0.05, d["mean"].max()) * 1.3)
plt.tight_layout(); plt.savefig(f"{RES}/figs/ablations.png", dpi=200)

# Figure 3: failure types per town
ft_town[["matched_%", "misplaced_%", "missing_%"]].plot.bar(stacked=True, figsize=(6, 4), color=["#1E6B57", "#C8642A", "#B3392B"])
plt.ylabel("% of real road length"); plt.title("How OpenStreetMap covers the real roads"); plt.tight_layout()
plt.savefig(f"{RES}/figs/failure_types.png", dpi=200); plt.show()

In [ ]:
# 9. Flags: precision and recall per town, with the main model (RQ2)
main_model = new_model(); main_model.load_state_dict(torch.load(f"{RES}/main_model.pt"))
thr = pick_threshold(main_model)
flag_rows, seg_rows = [], []
for n, p in zip(test_n, probs(main_model, test_n)):
    t = tiles[n]; fl, sg = flags_for_tile(p > thr, t["osm"], t["fixed"])
    flag_rows += [dict(tile=n, town=t["town"], **f) for f in fl]; seg_rows += [dict(tile=n, town=t["town"], **s) for s in sg]
F, S = pd.DataFrame(flag_rows), pd.DataFrame(seg_rows)
F.to_csv(f"{RES}/flags_test.csv", index=False); S.to_csv(f"{RES}/missing_segments_test.csv", index=False)
out = []
for town in sorted(set(F.town) | set(S.town)):
    f, s = F[F.town == town], S[S.town == town]
    good = int(f.kind.isin(["missing_road", "misplaced_osm"]).sum())
    P = wilson(good, len(f)); Rc = wilson(int(s.flagged.sum()), len(s))
    out.append(dict(town=town, flags=len(f), **{f"{k}_share": round((f.kind == k).mean(), 3) for k in ("missing_road", "misplaced_osm", "not_a_road")},
                    flag_precision=round(P[0], 3), precision_ci=f"{P[1]:.2f}-{P[2]:.2f}",
                    missing_segments=len(s), flag_recall=round(Rc[0], 3), recall_ci=f"{Rc[1]:.2f}-{Rc[2]:.2f}"))
flag_summary = pd.DataFrame(out); flag_summary.to_csv(f"{RES}/flag_precision_recall.csv", index=False); print(flag_summary)

# Gallery of flags for a quick manual check (merged roads cannot be detected automatically)
sample = F.sample(min(24, len(F)), random_state=0).reset_index(drop=True)
fig, axs = plt.subplots(4, 6, figsize=(18, 12))
for a, (_, f) in zip(axs.flat, sample.iterrows()):
    t = tiles[f.tile]; r0, c0 = max(0, f.row - 64), max(0, f.col - 64)
    a.imshow(t["img"][r0:r0 + 128, c0:c0 + 128]); a.contour(t["osm"][r0:r0 + 128, c0:c0 + 128], [0.5], colors="yellow")
    a.contour(t["fixed"][r0:r0 + 128, c0:c0 + 128], [0.5], colors="cyan"); a.plot(f.col - c0, f.row - r0, "r+", ms=14)
    a.set_title(f"#{_} {f.kind}", fontsize=9); a.axis("off")
plt.suptitle("Flags (red +). Yellow = OSM, cyan = corrected road"); plt.tight_layout(); plt.savefig(f"{RES}/figs/flag_gallery.png", dpi=120)
sample.to_csv(f"{RES}/flag_gallery_sample.csv")

In [ ]:
# 10. Precision-recall curve and failure analysis (3 worst test tiles) for the main model
pt_ = probs(main_model, test_n); pr = []
for th in np.arange(0.1, 0.95, 0.05):
    c = pd.DataFrame([counts(p > th, tiles[n]["fixed"]) for n, p in zip(test_n, pt_)]).sum()
    s = score(c); pr.append(dict(threshold=round(th, 2), precision=s["precision"], recall=s["recall"]))
pr = pd.DataFrame(pr); pr.to_csv(f"{RES}/pr_curve.csv", index=False)
plt.figure(figsize=(5, 4)); plt.plot(pr.recall, pr.precision, marker="o"); plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title(f"Main model, test tiles (chosen threshold {thr})"); plt.tight_layout(); plt.savefig(f"{RES}/figs/pr_curve.png", dpi=200)

per_tile = pd.DataFrame([dict(tile=n, **score(counts(p > thr, tiles[n]["fixed"]))) for n, p in zip(test_n, pt_)])
worst = per_tile.sort_values("f1").head(3)
fig, axs = plt.subplots(3, 4, figsize=(14, 10.5))
for row, (_, w) in zip(axs, worst.iterrows()):
    t = tiles[w.tile]; p = pt_[test_n.index(w.tile)] > thr
    for a, im, ti in zip(row, [t["img"], t["osm"], t["fixed"], p], ["Image", "OSM", "Corrected label", f"Prediction (F1 {w.f1:.2f})"]):
        a.imshow(im, cmap=None if ti == "Image" else "gray"); a.set_title(f"{ti}\n{w.tile}" if ti == "Image" else ti, fontsize=9); a.axis("off")
plt.tight_layout(); plt.savefig(f"{RES}/figs/failure_cases.png", dpi=150); per_tile.to_csv(f"{RES}/per_tile_main.csv", index=False)

In [ ]:
# 11. Download everything (result files behind every chart, for the GitHub repo)
!cd "{RES}" && zip -q -r /content/final_results.zip . -x "*.pt"
from google.colab import files; files.download("/content/final_results.zip")